# Welcome to GNM!

This Colab demonstrates the [GNM head model](https://github.com/google/GNM) and visualizes the effects of its parameters.

### How do I run this?


In the menu-bar above, click **Runtime &rarr; Run all**.

In [ ]:
# @title Import dependencies.
import itertools

import re

from IPython.display import display
import ipywidgets as widgets
import numpy as np
from scipy.spatial.transform import Rotation

from gnm.shape import gnm_numpy
from gnm.shape import gnm_jupyter_viewer

GNMMeshViewer = gnm_jupyter_viewer.GNMMeshViewer


In [ ]:
# @title Instantiate GNM

gnm = gnm_numpy.GNM.from_local(
    version=gnm_numpy.GNMMajorVersion.V3,
    variant=gnm_numpy.GNMVariant.HEAD
)


In [ ]:
# @title Define a GUI to control GNM parameters.
# @markdown This cell contains a lot of boilerplate GUI code. Just run it to get to the demo.

# A thin gray line to separate horizontal components.
divider = widgets.HTML(
    "<div style='height:100%;border-left:2px solid #EEE;margin:0 1.1em'></div>"
)

# Shared arguments for vertical sliders.
vslider_kwargs = dict(
    orientation="vertical", layout=widgets.Layout(width="2.2em")
)

# Shared layout for vertical boxes.
vlayout = widgets.Layout(align_items="center", justify_content="space-between")


# Creates title text.
def title(name: str) -> None:
  return widgets.HTML(f"<div style='font-size:1.4em;'><b>{name}</b></div>")


# Creates label text.
def label(name: str) -> None:
  html = f"<div style='font-size:1.1em;line-height:100%'><b>{name}</b></div>"
  return widgets.HTML(html)


# Clicking this button will reset sliders to 0.0.
def reset_button(sliders):
  button = widgets.Button(description="Reset")
  button.layout.width = "6em"

  def click(unused):
    for slider in sliders:
      slider.value = 0.0

  button.on_click(click)
  return button


# Groups sliders with a title, and adds a reset button.
def slider_group(label, sliders):
  box = widgets.HBox if sliders[0].orientation == "vertical" else widgets.VBox
  gui = widgets.VBox([label, box(sliders), reset_button(sliders)])
  gui.layout = vlayout
  return gui


def _find_component_indices(
    names: list[str], region: str, suffix: str = "_[0-9][0-9][0-9]"
) -> list[int]:
  regex = re.compile(f"{region}{suffix}")
  find_func = np.vectorize(lambda x: bool(regex.match(x)))
  return np.where(find_func(names))[0]


class GNMDemoGui:

  def __init__(self, gnm: gnm_numpy.GNM):

    self.gnm = gnm

    # Build Identity GUI
    head_indices = _find_component_indices(gnm.identity_names, "head")
    self.identity_indices = {
        "head": head_indices[:10],
    }
    self.identity_sliders = {
        k: [
            widgets.FloatSlider(
                description=f"i{i}",
                min=-3.0,
                max=3.0,
                orientation="vertical",
                layout=widgets.Layout(width="2.2em", height="13em"),
            )
            for i in v
        ]
        for k, v in self.identity_indices.items()
    }
    identity_slider_groups = [
        slider_group(label(k), v) for k, v in self.identity_sliders.items()
    ]
    identity_gui_elements = list(
        itertools.chain(*[
            [slider_group, divider] for slider_group in identity_slider_groups
        ])
    )[:-1]
    identity_gui = widgets.VBox(
        [title("Identity"), widgets.HBox(identity_gui_elements)]
    )
    identity_gui.layout = vlayout

    # Build Expression GUI
    left_eye_indices = _find_component_indices(
      gnm.expression_names, "left_eye"
    )
    right_eye_indices = _find_component_indices(
      gnm.expression_names, "right_eye"
    )
    mouth_indices = _find_component_indices(gnm.expression_names, "mouth")
    tongue_mean_indices = _find_component_indices(
        gnm.expression_names, "tongue_mean", suffix=""
    )
    tongue_indices = _find_component_indices(gnm.expression_names, "tongue")
    all_tongue_indices = np.concatenate(
        [tongue_mean_indices, tongue_indices], axis=0
    )
    eyeballs_indices = _find_component_indices(gnm.expression_names, "eyeballs")
    self.expression_indices = {
        "left eye": left_eye_indices[:3],
        "right eye": right_eye_indices[:3],
        "mouth": mouth_indices[:7],
        "tongue": all_tongue_indices[:4],
        "pupil": eyeballs_indices[:1],
    }
    self.expression_sliders = {
        k: [
            widgets.FloatSlider(
                description=f"e{i}",
                min=-3.0,
                max=3.0,
                orientation="vertical",
                layout=widgets.Layout(width="2.2em", height="13em"),
            )
            for i in v
        ]
        for k, v in self.expression_indices.items()
    }
    expression_slider_groups = [
        slider_group(label(k), v) for k, v in self.expression_sliders.items()
    ]
    expression_gui_elements = list(
        itertools.chain(*[
            [slider_group, divider] for slider_group in expression_slider_groups
        ])
    )[:-1]

    expression_gui = widgets.VBox(
        [title("Expression"), widgets.HBox(expression_gui_elements)]
    )
    expression_gui.layout = vlayout

    # Build Pose GUI
    neck_limits = {"X": 90, "Y": 90, "Z": 90}
    head_limits = {"X": 45, "Y": 45, "Z": 15}
    gaze_limits = {"X": 25, "Y": 30, "Verg.": 10}
    all_limits = {"neck": neck_limits, "head": head_limits, "gaze": gaze_limits}
    pose_gui_elements = []
    self.pose_sliders = {name: [] for name in all_limits}

    for item, limits in all_limits.items():
      for name, limit in limits.items():
        slider = widgets.IntSlider(
            description=name,
            min=-limit,
            max=limit,
            orientation="vertical",
            layout=widgets.Layout(width="2.2em", height="13em"),
        )
        self.pose_sliders[item].append(slider)

      gui = slider_group(label(item.capitalize()), self.pose_sliders[item])
      pose_gui_elements += [gui, divider]

    pose_gui = widgets.VBox(
        [title("Pose"), widgets.HBox(pose_gui_elements[:-1])]
    )
    pose_gui.layout = vlayout

    # Build translation GUI
    self.translation_sliders = [
        widgets.FloatSlider(
            description=d, min=-0.2, max=0.2, step=0.01, **vslider_kwargs
        )
        for d in ["X", "Y", "Z"]
    ]
    translation_gui = slider_group(
        title("Translation"), self.translation_sliders
    )

    gui = widgets.HBox([
        identity_gui,
        divider,
        expression_gui,
        divider,
        pose_gui,
        divider,
        translation_gui,
    ])
    gui.layout.margin = "0 0 1em 0"
    display(gui)

  @property
  def sliders(self):
    all_pose_sliders = sum([list(l) for l in self.pose_sliders.values()], [])
    identity_sliders = list(itertools.chain(*self.identity_sliders.values()))
    expression_sliders = list(
        itertools.chain(*self.expression_sliders.values())
    )
    return (
        identity_sliders
        + expression_sliders
        + all_pose_sliders
        + self.translation_sliders
    )

  def get_identity_parameters(self):
    identity = np.zeros(self.gnm.identity_dim)
    for region in self.identity_indices.keys():
      for i, slider in zip(
          self.identity_indices[region], self.identity_sliders[region]
      ):
        identity[i] = slider.value
    return identity

  def get_expression_parameters(self):
    expression = np.zeros(self.gnm.expression_dim)
    for region in self.expression_indices.keys():
      for i, slider in zip(
          self.expression_indices[region], self.expression_sliders[region]
      ):
        expression[i] = slider.value
    return expression

  def get_rotation_parameters(self):

    # First, build the rotations for neck and head joints.
    rotations_euler = np.zeros((self.gnm.num_joints, 3))
    for i, joint_name in enumerate(self.gnm.joint_names[:2]):
      sliders = gui.pose_sliders[joint_name]
      for j, slider in enumerate(sliders):
        rotations_euler[i, j] = np.radians(slider.value)

    # Rather than allow the user to control each eye separately, we keep both
    # eyes locked together, and expose a separate vergence control. Rotation
    # about the Z-axis is meaningless for the eyeballs anyway.
    gaze_sliders = gui.pose_sliders["gaze"]
    for i in range(2):
      rotations_euler[2:4, i] = np.radians(gaze_sliders[i].value)

    # Assign the gaze vergence.
    rotations_euler[2, 1] += np.radians(gaze_sliders[2].value)
    rotations_euler[3, 1] -= np.radians(gaze_sliders[2].value)

    rotations_aa = np.array(
        [Rotation.from_euler("XYZ", r).as_rotvec() for r in rotations_euler]
    )

    return rotations_aa

  def get_translation_parameters(self):
    return np.array([s.value for s in self.translation_sliders])

###Run the cell below to launch the demo.

We expose some GNM parameters:
* The first ten components of the linear identity model.
* The first nine components of the linear expression model, three per head region (left eye, right eye, mouth).
* Joint rotations for the neck, head, and eyeballs.
* Translation.

Move any slider and see the effect it has on GNM.

In [ ]:

gui = GNMDemoGui(gnm)
viewer = GNMMeshViewer(gnm)

def update_mesh(unused):

  # Extract GNM parameters from GUI.
  identity = gui.get_identity_parameters()
  expression = gui.get_expression_parameters()
  rotations = gui.get_rotation_parameters()
  translation = gui.get_translation_parameters()

  # Evaluate the GNM mesh generating function.
  vertices = gnm(identity, expression, rotations, translation)

  viewer.update(vertices)


for slider in gui.sliders:
  slider.observe(update_mesh)